In [3]:
import json
from pathlib import Path
from datasets import load_dataset

# Your exact absolute paths
OUTPUT_DIR = Path("/Users/farhanwajid/Web Development/SIH/SatQuery-AI---An-Interactive-Vision-Language-Assistant-for-Multimodal-Remote-Sensing-Image-Analysis-/FineTuning-VLM-LLM/model_training/data_prep/Outputs")
OUTPUT_FILE = OUTPUT_DIR / "rsvqa_train.jsonl"
IMAGE_CACHE_DIR = Path("/Users/farhanwajid/Web Development/SIH/downloaded_benchmarks/rsvqa_cache")

# Ensure directories exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("📥 Streaming RSVQA-LR from active Hugging Face mirror (dmarsili/RSVQA-LR-2k)...")

try:
    # Fix: Changed split="train" to split="validation"
    dataset = load_dataset("dmarsili/RSVQA-LR-2k", split="validation", streaming=True)
    
    samples = []
    MAX_SAMPLES = 2000  

    for idx, item in enumerate(dataset):
        if idx >= MAX_SAMPLES:
            break

        # 1. Extract standard fields
        question = item.get("question", "Describe the features in this satellite view.")
        
        # Failsafe: Handle if the answer is stored as a list or a raw string
        raw_answer = item.get("answer", item.get("answers", "No answer provided."))
        answer = raw_answer[0] if isinstance(raw_answer, list) else raw_answer
        
        # 2. Save image locally
        img_path = IMAGE_CACHE_DIR / f"rsvqa_{idx}.png"
        if not img_path.exists() and "image" in item:
            item["image"].save(img_path)

        # 3. Format into Qwen2-VL Schema
        entry = {
            "id": f"rsvqa_{idx}",
            "images": [str(img_path.resolve())],
            "conversations": [
                {
                    "role": "user",
                    "content": f"<|vision_start|><|image_pad|><|vision_end|>\n{question}"
                },
                {
                    "role": "assistant",
                    "content": str(answer)
                }
            ]
        }
        samples.append(entry)
        
        if (idx + 1) % 500 == 0:
            print(f"  • Processed {idx + 1} / {MAX_SAMPLES} samples...")

    # 4. Save to JSONL
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for entry in samples:
            f.write(json.dumps(entry) + "\n")

    print(f"\n✅ Prepared {len(samples)} RSVQA instruction pairs at: {OUTPUT_FILE}")

except Exception as e:
    print(f"⚠️ Error occurred: {e}")

📥 Streaming RSVQA-LR from active Hugging Face mirror (dmarsili/RSVQA-LR-2k)...
  • Processed 500 / 2000 samples...
  • Processed 1000 / 2000 samples...
  • Processed 1500 / 2000 samples...
  • Processed 2000 / 2000 samples...

✅ Prepared 2000 RSVQA instruction pairs at: /Users/farhanwajid/Web Development/SIH/SatQuery-AI---An-Interactive-Vision-Language-Assistant-for-Multimodal-Remote-Sensing-Image-Analysis-/FineTuning-VLM-LLM/model_training/data_prep/Outputs/rsvqa_train.jsonl
